# 20.5 換個問法，為什麼仍可能是同一道舊題？


剛才那張貓照片，問數量和問姿勢是兩道問題，卻仍是同一張照片。如果一題拿來訓練，另一題算成測試，模型可能已經認熟那張桌子與兩隻貓。我們需要先分素材，再生題目。

[5.12](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)把這種關聯叫資料家族：同一份來源的裁切、換問法、翻譯或其他變體，不應各自抽籤到不同考卷。本章使用的DOCCI照片資料集提供公開的視覺相似群組：群內圖片看起來相近，但不等於已核實同一拍攝場次，也不保證不同群來自不同攝影者。聊天樹則有共享前文的多個分支。切分單位要跟真正會共享線索的來源相符。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural-v4-family-crops.svg")))

用四張手寫卡片看一個最小例子：


In [ ]:
records = [
    {"family": "貓照片A", "split": "train", "question": "有幾隻貓？"},
    {"family": "貓照片A", "split": "train", "question": "哪隻貓抬起前腳？"},
    {"family": "招牌照片B", "split": "test", "question": "招牌寫什麼？"},
    {"family": "招牌照片B", "split": "test", "question": "有沒有中文字？"},
]
families = {name: {row["family"] for row in records if row["split"] == name} for name in ["train", "test"]}
print("訓練家族", sorted(families["train"]))
print("測試家族", sorted(families["test"]))
print("跨份重疊", sorted(families["train"] & families["test"]))

每筆卡片有家族、用途與問題。建立`families`時，外層先後取`train`與`test`作為字典的鍵；內層用`if`挑出對應用途的卡片，再收集它們的`family`欄位。集合把重複家族合成一項，`&`取兩邊共同的家族，所以最後是空清單。這只核對手寫卡片；正式資料另外保存來源ID與檔案指紋，防止同一圖片只換名字便跨到另一側。

訓練題用來更新權重，驗證題用來選設定，最後測試則留到版本固定以後才打開。已公開、看過答案的舊題可以作回歸檢查，觀察以前能力是否保留；它們不再是新一輪未看過的最後考卷。這裡的公開底座，是模型製作者已訓練好、提供大家使用的模型；製作者就是我們說的上游。本課在底座上繼續訓練稱為微調，再用驗證結果選定交付版本。家族切分隔開的是本課的訓練、驗證與最後測試；製作者先前可能用過這些公開素材，因此不能宣稱模型從未見過。

練習把第二張卡的`split`改成`test`。最後就會列出貓照片A，即使兩題文字不同。修正是把整個家族一起分配，不是替第二張卡改一個新家族名字。保留好這條邊界，接著才能談模型究竟改了誰。
